# Séance 12 · Exercices — Les agents, et le projet final · ⭐⭐⭐

**Niveau : ⭐⭐⭐ Avancé**

**Niveau de la séance : ⭐⭐⭐** · chaque exercice porte son propre niveau (⭐ Débutant · ⭐⭐ Intermédiaire · ⭐⭐⭐ Avancé).

Comment travailler : lis l'énoncé, code dans la cellule « À toi », lance la cellule de vérification (✅ / ❌), et n'ouvre la solution qu'après avoir vraiment essayé.
Tout tourne dans **Google Colab**, rien à installer. Exécute chaque cellule avec `Maj + Entrée`.
Tous les exercices marchent en **mode démo** (`USE_MODEL = False`) : le faux modèle suit le protocole `OUTIL: nom(arguments)` dès que le prompt système le décrit. Ce qu'on vérifie, c'est **ta boucle d'agent**, pas le talent du modèle.


## Préparation

La même cellule qu'à la leçon (elle prépare `llm(messages)`), plus le fichier Pokémon (l'outil 1), la calculatrice `_evaluer` de la leçon (la partie délicate de l'outil 2, fournie), l'outil 3 `chercher_dans_mes_notes` déjà prêt, et le helper `verifier`. Lance-la une fois.

In [ ]:
USE_MODEL = True   # ← mets False pour tester le notebook sans modèle (réponses factices, sans GPU)

import json, re
import ast, operator
import pandas as pd

# ---------- Mode démo : un faux LLM qui répond sans réseau ni GPU ----------
def llm_factice(messages):
    """Mode démo : un faux modèle qui suit le protocole OUTIL: nom(args) quand le prompt système le décrit."""
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
    question = next(m["content"] for m in messages if m["role"] == "user")      # la question de départ
    ql = question.lower()
    if "OUTIL:" in systeme:                                                      # ---- mode agent ----
        resultats = [m["content"].split(":", 1)[1].split("\n")[0].strip()
                     for m in messages if m["role"] == "user" and m["content"].startswith("Résultat de l'outil")]
        if resultats:                                                            # on a déjà un résultat
            stat = re.search(r"vitesse|attaque|défense|pv", ql)
            facteur = re.search(r"(?:fois|multipli\w+ par|x|×)\s*(\d+)", ql)
            valeur = re.search((stat.group(0) if stat else "vitesse") + r"\D*(\d+)", resultats[-1].lower())
            if len(resultats) == 1 and facteur and valeur:                      # « la vitesse de X fois 2 » → calcul
                return f"OUTIL: calculer({valeur.group(1)} * {facteur.group(1)})"
            return "D'après mes outils : " + " ; ".join(resultats)
        expression = re.search(r"\d[\d\s+\-*/x×().]*\d", question)
        if expression and re.search(r"\d\s*[-+*/x×]\s*\d", question):
            return f"OUTIL: calculer({expression.group(0).replace('x', '*').replace('×', '*').strip()})"
        noms = re.findall(r"\b[A-Z][a-zé]{2,}\b", question[1:])                 # mots avec majuscule (sauf le 1er)
        if noms and re.search(r"pok[ée]mon|vitesse|attaque|défense|pv|type|rapide|fort", ql):
            return f"OUTIL: chercher_pokemon({noms[0]})"
        if re.search(r"notes|cours|séance|token|température|rag|api|rôle|agent|embedding|json", ql):
            return f"OUTIL: chercher_dans_mes_notes({question})"
        return "Je n'ai pas besoin d'outil : bonjour, je suis ton assistant !"
    if "questions" in ql and "invités" in ql:                                    # projet final
        return ("1. Qu'est-ce que ton programme fait exactement ? "
                "2. Qu'est-ce qui a été le plus difficile ? "
                "3. Est-ce que l'IA pourrait se tromper, et comment tu le saurais ?")
    if re.search(r"\d\s*[-+*/x×]\s*\d", ql):                                     # calcul sans outil : il se trompe
        return "Le résultat est 9 156."
    if re.search(r"pok[ée]mon|vitesse|attaque|pv", ql):                          # Pokémon sans outil : il invente
        return "Pikachu est un Pokémon Électrique avec 60 PV et une vitesse de 120."
    return "Bonne question ! Je ne suis pas certain, mais voici une réponse plausible : c'est un sujet intéressant."

# ---------- Le vrai modèle : petit modèle ouvert, gratuit, sans clé ----------

if USE_MODEL:
    %pip install -q transformers accelerate
    from transformers import pipeline
    _pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def llm(messages, max_new_tokens=150, temperature=0.7):
    """Envoie une liste de messages au modèle et renvoie sa réponse (du texte)."""
    if not USE_MODEL:
        return llm_factice(messages)
    if temperature == 0:      # température 0 = toujours la réponse la plus probable
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    return sortie[0]["generated_text"][-1]["content"].strip()

print("Modèle prêt :", "Qwen2.5-0.5B-Instruct" if USE_MODEL else "mode démo (llm_factice)")

# ---------- Outils des exercices ----------
URL_POKEMON = "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv"
try:
    pokemon = pd.read_csv(URL_POKEMON)
except Exception as e:
    print("Pas de réseau ?", e, "→ on utilise 3 Pokémon de secours")
    pokemon = pd.DataFrame([
        {"Name": "Pikachu", "Type 1": "Electric", "HP": 35, "Attack": 55, "Defense": 40, "Speed": 90},
        {"Name": "Charizard", "Type 1": "Fire", "HP": 78, "Attack": 84, "Defense": 78, "Speed": 100},
        {"Name": "Snorlax", "Type 1": "Normal", "HP": 160, "Attack": 110, "Defense": 65, "Speed": 30},
    ])

# La partie délicate de la calculatrice sécurisée (leçon, section 2) : on n'accepte que des nombres et + - * / ** %
OPERATIONS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
              ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod, ast.USub: operator.neg}

def _evaluer(noeud):
    if isinstance(noeud, ast.Constant) and isinstance(noeud.value, (int, float)):
        return noeud.value
    if isinstance(noeud, ast.UnaryOp) and type(noeud.op) in OPERATIONS:
        return OPERATIONS[type(noeud.op)](_evaluer(noeud.operand))
    if isinstance(noeud, ast.BinOp) and type(noeud.op) in OPERATIONS:
        gauche, droite = _evaluer(noeud.left), _evaluer(noeud.right)
        if isinstance(noeud.op, ast.Pow) and abs(droite) > 100:
            raise ValueError("exposant trop grand")
        return OPERATIONS[type(noeud.op)](gauche, droite)
    raise ValueError(f"élément interdit : {type(noeud).__name__}")

# Outil 3, déjà prêt : le RAG de la séance 11 en version courte
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

MES_NOTES = """Séance 9 : un token est un morceau de mot transformé en nombre. Le modèle ne voit pas les lettres, c'est pour ça qu'il compte mal les r de strawberry.

Séance 9 : un LLM fait une seule chose, prédire le token suivant, encore et encore. La température règle le hasard : 0 = toujours pareil, élevée = créatif puis délirant.

Séance 9 : un LLM hallucine, c'est-à-dire qu'il invente une réponse plausible quand il ne sait pas. Il a une date de connaissance et il calcule mal.

Séance 10 : une API, c'est comme un serveur de restaurant. On envoie une commande (la requête) et on reçoit un plat (la réponse), souvent en JSON.

Séance 10 : les trois rôles sont system (les consignes), user (l'utilisateur) et assistant (le modèle). Le prompt système donne la personnalité et les règles.

Séance 10 : un chatbot n'a pas de mémoire, il renvoie tout l'historique des messages à chaque tour. Pour réutiliser une réponse dans un programme, on demande du JSON.

Séance 11 : le RAG donne au modèle les bons passages de mes documents avant de poser la question. Les étapes sont découper, vectoriser, chercher, injecter, répondre."""
notes_chunks = [p.strip() for p in MES_NOTES.split("\n\n") if p.strip()]
STOP_FR = "le la les l un une des du de d et ou à a au aux en dans sur par pour avec ce cet cette ces se son sa ses mon ma mes ton ta tes il elle on ne pas plus que qui quoi quel quelle quels est sont c'est".split()
tfidf = TfidfVectorizer(stop_words=STOP_FR).fit(notes_chunks)
notes_vecteurs = tfidf.transform(notes_chunks)

def chercher_dans_mes_notes(question, k=2):
    """Renvoie les k passages de mes notes les plus proches de la question."""
    scores = cosine_similarity(tfidf.transform([question]), notes_vecteurs)[0]
    meilleurs = [i for i in scores.argsort()[::-1][:k] if scores[i] > 0.05]
    if not meilleurs:
        return "Rien trouvé dans les notes."
    return " | ".join(notes_chunks[i] for i in meilleurs)

# ---------- Vérification automatique des exercices ----------
def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais planter. `condition` = un booléen, ou une fonction sans argument."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as e:
        ok = False
        print(f"   (erreur pendant la vérification : {type(e).__name__} : {e})")
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else "  → pas encore, relis l'énoncé et réessaie"))

print("Prêt :", len(pokemon), "Pokémon,", len(notes_chunks), "paragraphes de notes")

## Exercice 1 ⭐ · Repérer un appel d'outil

Le protocole : quand le modèle veut un outil, il écrit `OUTIL: nom(arguments)`. Écris `extraire_appel(reponse)` qui renvoie le couple `(nom, arguments)` s'il y a un appel dans la réponse, sinon `None`. Sois tolérant : `outil : ...` en minuscules, et une phrase autour (« Je vais utiliser OUTIL: calculer(2+2) pour ça. »).

Résultat attendu : `("chercher_pokemon", "Pikachu")`, `("calculer", "348 * 27")`, `None`, `("calculer", "2+2")`, `("calculer", "2+2")`.

<details><summary>Indice</summary>

`re.search(r"OUTIL\s*:\s*(\w+)\((.*?)\)", reponse, re.IGNORECASE)` : `(\w+)` attrape le nom, `(.*?)` les arguments jusqu'à la première parenthèse fermante. Puis `appel.group(1)`, `appel.group(2).strip()`.

</details>

In [ ]:
# À toi
def extraire_appel(reponse):
    return None

essais = [
    "OUTIL: chercher_pokemon(Pikachu)",
    "OUTIL: calculer(348 * 27)",
    "Bonjour, je suis ton assistant !",
    "outil : calculer(2+2)",
    "Je vais utiliser OUTIL: calculer(2+2) pour ça.",
]
for e in essais:
    print(f"{e!r:50} → {extraire_appel(e)}")

In [ ]:
verifier("Exercice 1 · appels simples", lambda: extraire_appel(essais[0]) == ("chercher_pokemon", "Pikachu") and extraire_appel(essais[1]) == ("calculer", "348 * 27"))
verifier("Exercice 1 · pas d'appel → None", lambda: extraire_appel(essais[2]) is None)
verifier("Exercice 1 · tolérant", lambda: extraire_appel(essais[3]) == ("calculer", "2+2") and extraire_appel(essais[4]) == ("calculer", "2+2"))

<details><summary>Solution</summary>

```python
def extraire_appel(reponse):
    appel = re.search(r"OUTIL\s*:\s*(\w+)\((.*?)\)", reponse, re.IGNORECASE | re.DOTALL)
    return (appel.group(1), appel.group(2).strip()) if appel else None

essais = [
    "OUTIL: chercher_pokemon(Pikachu)",
    "OUTIL: calculer(348 * 27)",
    "Bonjour, je suis ton assistant !",
    "outil : calculer(2+2)",
    "Je vais utiliser OUTIL: calculer(2+2) pour ça.",
]
for e in essais:
    print(f"{e!r:50} → {extraire_appel(e)}")
```

</details>

## Exercice 2 ⭐ · Le premier outil : chercher un Pokémon

Un outil = une fonction qui prend du **texte** et renvoie du **texte**. Écris `chercher_pokemon(nom)` qui cherche le Pokémon dans le tableau `pokemon` (majuscules et espaces ignorés) et renvoie `"Nom : type T, H PV, attaque A, défense D, vitesse S."`, ou `"Aucun Pokémon appelé X."` s'il n'existe pas.

Résultat attendu : `chercher_pokemon("pikachu")` → `"Pikachu : type Electric, 35 PV, attaque 55, défense 40, vitesse 90."`

<details><summary>Indice</summary>

`ligne = pokemon[pokemon["Name"].str.lower() == nom.strip().lower()]`, puis `if ligne.empty: ...`, sinon `p = ligne.iloc[0]` et une f-string avec `p["Name"]`, `p["Type 1"]`, `p["HP"]`, `p["Attack"]`, `p["Defense"]`, `p["Speed"]`.

</details>

In [ ]:
# À toi
def chercher_pokemon(nom):
    return None

print(chercher_pokemon("pikachu"))
print(chercher_pokemon("Pikachou"))

In [ ]:
verifier("Exercice 2 · Pikachu", lambda: chercher_pokemon(" pikachu ") == "Pikachu : type Electric, 35 PV, attaque 55, défense 40, vitesse 90.")
verifier("Exercice 2 · inconnu", lambda: chercher_pokemon("Pikachou").startswith("Aucun Pokémon appelé Pikachou"))

<details><summary>Solution</summary>

```python
def chercher_pokemon(nom):
    ligne = pokemon[pokemon["Name"].str.lower() == nom.strip().lower()]
    if ligne.empty:
        return f"Aucun Pokémon appelé {nom.strip()}."
    p = ligne.iloc[0]
    return f"{p['Name']} : type {p['Type 1']}, {p['HP']} PV, attaque {p['Attack']}, défense {p['Defense']}, vitesse {p['Speed']}."

print(chercher_pokemon("pikachu"))
print(chercher_pokemon("Pikachou"))
```

</details>

## Exercice 3 ⭐ · Le dispatcher : du nom à la fonction

Les outils sont rangés dans un dictionnaire `OUTILS` (nom → fonction). Écris `executer(nom, args)` qui appelle la bonne fonction avec `args` et renvoie son résultat, ou, si le nom est inconnu, le texte `"Outil inconnu : nom. Outils disponibles : [...]"` (sans planter !).

Résultat attendu : `executer("chercher_pokemon", "Pikachu")` contient « vitesse 90 » ; `executer("voler", "Pikachu")` commence par « Outil inconnu : voler ».

<details><summary>Indice</summary>

`if nom not in OUTILS: return f"Outil inconnu : {nom}. Outils disponibles : {list(OUTILS)}"`, sinon `return OUTILS[nom](args)`.

</details>

In [ ]:
# À toi
OUTILS = {"chercher_pokemon": chercher_pokemon, "chercher_dans_mes_notes": chercher_dans_mes_notes}

def executer(nom, args):
    return None

print(executer("chercher_pokemon", "Pikachu"))
print(executer("chercher_dans_mes_notes", "c'est quoi un token"))
print(executer("voler", "Pikachu"))

In [ ]:
verifier("Exercice 3 · outil connu", lambda: "vitesse 90" in executer("chercher_pokemon", "Pikachu") and "token" in executer("chercher_dans_mes_notes", "c'est quoi un token"))
verifier("Exercice 3 · outil inconnu", lambda: executer("voler", "Pikachu").startswith("Outil inconnu : voler") and "chercher_pokemon" in executer("voler", "Pikachu"))

<details><summary>Solution</summary>

```python
OUTILS = {"chercher_pokemon": chercher_pokemon, "chercher_dans_mes_notes": chercher_dans_mes_notes}

def executer(nom, args):
    if nom not in OUTILS:
        return f"Outil inconnu : {nom}. Outils disponibles : {list(OUTILS)}"
    return OUTILS[nom](args)

print(executer("chercher_pokemon", "Pikachu"))
print(executer("chercher_dans_mes_notes", "c'est quoi un token"))
print(executer("voler", "Pikachu"))
```

</details>

## Exercice 4 ⭐ · Décrire les outils au modèle

Le modèle ne connaît **que** les outils qu'on lui décrit dans le prompt système. Écris `decrire_outils(descriptions)` qui, à partir du dictionnaire `nom → (argument, description, exemple)`, renvoie une ligne par outil au format :
`- nom(argument) : description. Ex : OUTIL: nom(exemple)`
puis assemble `SYSTEME_AGENT` = `ENTETE` + la description + `REGLE` (séparés par des retours à la ligne).

Résultat attendu : 3 lignes, une par outil, et un `SYSTEME_AGENT` qui contient `OUTIL:` (c'est ce mot-clé qui déclenche le mode agent du faux modèle).

<details><summary>Indice</summary>

`lignes = [f"- {nom}({arg}) : {desc}. Ex : OUTIL: {nom}({ex})" for nom, (arg, desc, ex) in descriptions.items()]` puis `"\n".join(lignes)`.

</details>

In [ ]:
# À toi
DESCRIPTIONS = {
    "chercher_pokemon":        ("nom", "les statistiques d'un Pokémon (PV, attaque, défense, vitesse)", "Pikachu"),
    "calculer":                ("expression", "calcule une expression mathématique", "348 * 27"),
    "chercher_dans_mes_notes": ("question", "cherche dans mes notes de cours", "c'est quoi un token"),
}
ENTETE = "Tu es un assistant qui peut utiliser des outils. Tes outils :"
REGLE = """Pour utiliser un outil, réponds EXACTEMENT une ligne de la forme OUTIL: nom(arguments), et rien d'autre.
Quand tu reçois le résultat de l'outil, réponds à la question en français, en une phrase, à partir de ce résultat.
Si tu n'as pas besoin d'outil, réponds directement en une phrase."""

def decrire_outils(descriptions):
    return None

SYSTEME_AGENT = None
print(SYSTEME_AGENT)

In [ ]:
verifier("Exercice 4 · decrire_outils", lambda: decrire_outils(DESCRIPTIONS).split("\n")[1] == "- calculer(expression) : calcule une expression mathématique. Ex : OUTIL: calculer(348 * 27)" and len(decrire_outils(DESCRIPTIONS).split("\n")) == 3)
verifier("Exercice 4 · SYSTEME_AGENT", lambda: SYSTEME_AGENT.startswith(ENTETE) and SYSTEME_AGENT.endswith(REGLE) and "OUTIL: chercher_pokemon(Pikachu)" in SYSTEME_AGENT)

<details><summary>Solution</summary>

```python
DESCRIPTIONS = {
    "chercher_pokemon":        ("nom", "les statistiques d'un Pokémon (PV, attaque, défense, vitesse)", "Pikachu"),
    "calculer":                ("expression", "calcule une expression mathématique", "348 * 27"),
    "chercher_dans_mes_notes": ("question", "cherche dans mes notes de cours", "c'est quoi un token"),
}
ENTETE = "Tu es un assistant qui peut utiliser des outils. Tes outils :"
REGLE = """Pour utiliser un outil, réponds EXACTEMENT une ligne de la forme OUTIL: nom(arguments), et rien d'autre.
Quand tu reçois le résultat de l'outil, réponds à la question en français, en une phrase, à partir de ce résultat.
Si tu n'as pas besoin d'outil, réponds directement en une phrase."""

def decrire_outils(descriptions):
    lignes = [f"- {nom}({arg}) : {desc}. Ex : OUTIL: {nom}({ex})" for nom, (arg, desc, ex) in descriptions.items()]
    return "\n".join(lignes)

SYSTEME_AGENT = ENTETE + "\n" + decrire_outils(DESCRIPTIONS) + "\n" + REGLE
print(SYSTEME_AGENT)
```

</details>

## Exercice 5 ⭐⭐ · La calculatrice sécurisée

Jamais d'`eval` sur ce qu'écrit un modèle. La fonction `_evaluer` (fournie) sait calculer un arbre `ast` qui ne contient que des nombres et `+ - * / ** %`. Écris :
1. `calculer(expression)` : lit l'expression avec `ast.parse(expression.strip(), mode="eval")`, appelle `_evaluer(arbre.body)` et renvoie `"expression = résultat"` ; si **quoi que ce soit** rate, renvoie `"Expression refusée (raison)."` ;
2. `est_sure(expression)` : `True` si l'expression ne contient que des chiffres, espaces, `+ - * / % ( ) .` (un filtre rapide avant même de parser). Ajoute ensuite `calculer` à `OUTILS`.

Résultat attendu : `calculer("348 * 27")` → `"348 * 27 = 9396"` ; `calculer('__import__("os").system("echo piraté")')` → refusée ; `calculer("2 ** 1000")` → refusée (exposant trop grand).

<details><summary>Indice</summary>

`try: arbre = ast.parse(...); return f"{expression.strip()} = {_evaluer(arbre.body)}" except Exception as e: return f"Expression refusée ({e})."`. Pour `est_sure` : `re.fullmatch(r"[\d\s+\-*/%().]+", expression) is not None`.

</details>

In [ ]:
# À toi
def calculer(expression):
    return None

def est_sure(expression):
    return None

OUTILS["calculer"] = calculer

for e in ["348 * 27", "(90 + 100) / 2", "2 ** 1000", '__import__("os").system("echo piraté")']:
    print(f"{e!r:45} sûre : {est_sure(e)!s:5} → {calculer(e)}")

In [ ]:
verifier("Exercice 5 · calculs", lambda: calculer("348 * 27") == "348 * 27 = 9396" and calculer("(90 + 100) / 2") == "(90 + 100) / 2 = 95.0")
verifier("Exercice 5 · refus", lambda: calculer('__import__("os").system("echo piraté")').startswith("Expression refusée") and calculer("2 ** 1000").startswith("Expression refusée") and calculer("abc").startswith("Expression refusée"))
verifier("Exercice 5 · est_sure", lambda: est_sure("1 + 2") is True and est_sure("2 ** 8") is True and est_sure("import os") is False and est_sure('__import__("os")') is False)
verifier("Exercice 5 · outil ajouté", lambda: OUTILS["calculer"]("1 + 1") == "1 + 1 = 2")

<details><summary>Solution</summary>

```python
def calculer(expression):
    try:
        arbre = ast.parse(expression.strip(), mode="eval")
        return f"{expression.strip()} = {_evaluer(arbre.body)}"
    except Exception as e:                      # syntaxe cassée, élément interdit, exposant trop grand...
        return f"Expression refusée ({e})."

def est_sure(expression):
    return re.fullmatch(r"[\d\s+\-*/%().]+", expression) is not None

OUTILS["calculer"] = calculer

for e in ["348 * 27", "(90 + 100) / 2", "2 ** 1000", '__import__("os").system("echo piraté")']:
    print(f"{e!r:45} sûre : {est_sure(e)!s:5} → {calculer(e)}")
```

</details>

## Exercice 6 ⭐⭐ · La boucle de l'agent

Écris `agent(question, max_etapes=4, afficher=False, modele=llm)` :
1. `messages` = le prompt système `SYSTEME_AGENT` + la question ;
2. au plus `max_etapes` fois : `reponse = modele(messages, max_new_tokens=60, temperature=0)` ; si `extraire_appel(reponse)` ne trouve rien, c'est la **réponse finale** : renvoie-la ; sinon exécute l'outil avec `executer`, ajoute `reponse` à l'historique (rôle `assistant`) puis un message `user` **exactement** de la forme `f"Résultat de l'outil : {resultat}\nRéponds maintenant à la question."` ;
3. après `max_etapes` sans réponse finale, renvoie `f"Je n'ai pas réussi en {max_etapes} étapes."`.

Résultat attendu : « Quelle est la vitesse de Pikachu ? » → une réponse avec « 90 » ; « Combien font 348 * 27 ? » → « 9396 » ; « Bonjour, qui es-tu ? » → pas d'appel d'outil ; un faux modèle qui demande toujours un outil s'arrête après `max_etapes` appels.

<details><summary>Indice</summary>

Le squelette de la boucle : `for etape in range(1, max_etapes + 1): reponse = modele(...); appel = extraire_appel(reponse); if not appel: return reponse; nom, args = appel; resultat = executer(nom, args); messages.append(...); messages.append(...)`.

</details>

In [ ]:
# À toi
def agent(question, max_etapes=4, afficher=False, modele=llm):
    messages = [{"role": "system", "content": SYSTEME_AGENT}, {"role": "user", "content": question}]
    return None

print(agent("Quelle est la vitesse de Pikachu ?", afficher=True))
print(agent("Combien font 348 * 27 ?", afficher=True))
print(agent("Bonjour, qui es-tu ?"))

In [ ]:
verifier("Exercice 6 · outil Pokémon", lambda: "90" in agent("Quelle est la vitesse de Pikachu ?"))
verifier("Exercice 6 · outil calculer", lambda: "9396" in agent("Combien font 348 * 27 ?").replace(" ", ""))
verifier("Exercice 6 · sans outil", lambda: "OUTIL:" not in agent("Bonjour, qui es-tu ?"))
appels_infinis = []
def boucle_infinie(messages, **options):
    appels_infinis.append(1)
    return "OUTIL: calculer(1 + 1)"
verifier("Exercice 6 · max_etapes", lambda: agent("test", max_etapes=2, modele=boucle_infinie) == "Je n'ai pas réussi en 2 étapes." and len(appels_infinis) == 2)

<details><summary>Solution</summary>

```python
def agent(question, max_etapes=4, afficher=False, modele=llm):
    messages = [{"role": "system", "content": SYSTEME_AGENT}, {"role": "user", "content": question}]
    for etape in range(1, max_etapes + 1):
        reponse = modele(messages, max_new_tokens=60, temperature=0)
        appel = extraire_appel(reponse)
        if not appel:                                             # pas d'outil demandé : réponse finale
            return reponse
        nom, args = appel
        resultat = executer(nom, args)
        if afficher:
            print(f"  [étape {etape}] modèle → {reponse.strip()}")
            print(f"  [étape {etape}] outil  → {resultat}")
        messages.append({"role": "assistant", "content": reponse})
        messages.append({"role": "user", "content": f"Résultat de l'outil : {resultat}\nRéponds maintenant à la question."})
    return f"Je n'ai pas réussi en {max_etapes} étapes."

print(agent("Quelle est la vitesse de Pikachu ?", afficher=True))
print(agent("Combien font 348 * 27 ?", afficher=True))
print(agent("Bonjour, qui es-tu ?"))
```

</details>

## Exercice 7 ⭐⭐ · Ajouter un outil

Écris un 4e outil `plus_rapide(type_pokemon)` qui renvoie `"Le X le plus rapide est Nom (vitesse S)."` pour un type (`"Fire"`, `"Water"`...), ou `"Aucun Pokémon de type X."`. Puis ajoute-le **aux deux endroits** : le dictionnaire `OUTILS` (pour Python) et `DESCRIPTIONS` → `SYSTEME_AGENT` reconstruit (pour le modèle).

Résultat attendu : `plus_rapide("Fire")` cite le Pokémon de type Fire le plus rapide du fichier, et `SYSTEME_AGENT` contient une ligne `- plus_rapide(type) : ...`.

<details><summary>Indice</summary>

`du_type = pokemon[pokemon["Type 1"].str.lower() == type_pokemon.strip().lower()]`, puis `p = du_type.loc[du_type["Speed"].idxmax()]`. N'oublie pas `SYSTEME_AGENT = ENTETE + "\n" + decrire_outils(DESCRIPTIONS) + "\n" + REGLE`.

</details>

In [ ]:
# À toi
def plus_rapide(type_pokemon):
    return None

# ... ajoute l'outil à OUTILS, à DESCRIPTIONS, et reconstruis SYSTEME_AGENT

print(plus_rapide("Fire"))
print(plus_rapide("Licorne"))

In [ ]:
attendu_fire = pokemon[pokemon["Type 1"] == "Fire"].sort_values("Speed").iloc[-1]["Name"]
verifier("Exercice 7 · plus_rapide", lambda: attendu_fire in plus_rapide("fire") and plus_rapide("Licorne") == "Aucun Pokémon de type Licorne.")
verifier("Exercice 7 · ajouté à OUTILS", lambda: OUTILS["plus_rapide"] is plus_rapide)
verifier("Exercice 7 · décrit au modèle", lambda: "- plus_rapide(type)" in SYSTEME_AGENT and SYSTEME_AGENT.endswith(REGLE))

<details><summary>Solution</summary>

```python
def plus_rapide(type_pokemon):
    du_type = pokemon[pokemon["Type 1"].str.lower() == type_pokemon.strip().lower()]
    if du_type.empty:
        return f"Aucun Pokémon de type {type_pokemon.strip()}."
    p = du_type.loc[du_type["Speed"].idxmax()]
    return f"Le {p['Type 1']} le plus rapide est {p['Name']} (vitesse {p['Speed']})."

OUTILS["plus_rapide"] = plus_rapide                                        # pour Python
DESCRIPTIONS["plus_rapide"] = ("type", "le Pokémon le plus rapide d'un type", "Fire")   # pour le modèle
SYSTEME_AGENT = ENTETE + "\n" + decrire_outils(DESCRIPTIONS) + "\n" + REGLE

print(plus_rapide("Fire"))
print(plus_rapide("Licorne"))
# En mode démo, le faux modèle ne connaît pas plus_rapide ; avec un vrai modèle, essaie :
# agent("Quel est le Pokémon de type Fire le plus rapide ?", afficher=True)
```

</details>

## Exercice 8 ⭐⭐ · Un outil qui plante ne doit pas planter l'agent

Un outil peut lever une erreur (fichier absent, division par zéro...). Redéfinis `executer(nom, args)` pour qu'elle garde le comportement « outil inconnu » **et** attrape toute exception en renvoyant `"Erreur dans l'outil nom : message"`. Teste ensuite `agent` avec un faux modèle scripté qui demande d'abord un outil cassé, puis un outil inconnu, puis répond.

Résultat attendu : `executer("casse", "1")` commence par « Erreur dans l'outil casse », l'agent renvoie la réponse finale du script, et les deux messages « Résultat de l'outil » envoyés au modèle contiennent « Erreur » puis « Outil inconnu ».

<details><summary>Indice</summary>

`try: return OUTILS[nom](args) except Exception as e: return f"Erreur dans l'outil {nom} : {e}"`. Le faux modèle scripté : une liste de réponses et `reponses_script.pop(0)`, en notant `messages[-1]["content"]` dans `journal` à chaque appel.

</details>

In [ ]:
# À toi
OUTILS["casse"] = lambda args: 1 / 0          # un outil qui plante toujours

def executer(nom, args):
    return None

journal = []                                  # ce que le modèle reçoit à chaque appel
reponses_script = ["OUTIL: casse(1)", "OUTIL: voler(Pikachu)", "D'accord, je ne peux ni casser ni voler."]
def modele_scripte(messages, **options):
    journal.append(messages[-1]["content"])
    return reponses_script.pop(0)

print(executer("casse", "1"))
reponse_finale = agent("Fais n'importe quoi.", modele=modele_scripte)
print(reponse_finale)

In [ ]:
verifier("Exercice 8 · erreur attrapée", lambda: executer("casse", "1").startswith("Erreur dans l'outil casse") and executer("voler", "x").startswith("Outil inconnu") and "vitesse 90" in executer("chercher_pokemon", "Pikachu"))
verifier("Exercice 8 · l'agent survit", lambda: reponse_finale == "D'accord, je ne peux ni casser ni voler." and len(journal) == 3)
verifier("Exercice 8 · le modèle est informé", lambda: "Erreur dans l'outil casse" in journal[1] and "Outil inconnu : voler" in journal[2])

<details><summary>Solution</summary>

```python
OUTILS["casse"] = lambda args: 1 / 0

def executer(nom, args):
    if nom not in OUTILS:
        return f"Outil inconnu : {nom}. Outils disponibles : {list(OUTILS)}"
    try:
        return OUTILS[nom](args)
    except Exception as e:
        return f"Erreur dans l'outil {nom} : {e}"

journal = []
reponses_script = ["OUTIL: casse(1)", "OUTIL: voler(Pikachu)", "D'accord, je ne peux ni casser ni voler."]
def modele_scripte(messages, **options):
    journal.append(messages[-1]["content"])
    return reponses_script.pop(0)

print(executer("casse", "1"))
reponse_finale = agent("Fais n'importe quoi.", modele=modele_scripte)
print(reponse_finale)
# Un agent sérieux : chaque outil vérifie ses arguments, chaque erreur est rapportée au modèle, jamais de crash.
```

</details>

## Exercice 9 ⭐⭐⭐ · Le banc de test : le bon outil ?

Mesure ton agent : écris `outil_utilise(question)` qui envoie **un seul** appel au modèle (prompt système + question, `temperature=0`) et renvoie le nom du premier outil demandé (ou `None`), puis `score_agent(banc)` qui compte les questions où l'outil obtenu est celui attendu.

Résultat attendu (mode démo) : 6 / 6. Avec le vrai petit modèle, note ton score : c'est lui qui te dit si ton prompt système est clair.

<details><summary>Indice</summary>

`appel = extraire_appel(llm([...], max_new_tokens=60, temperature=0))` puis `appel[0] if appel else None`. Le score : `sum(outil_utilise(q) == attendu for q, attendu in banc)`.

</details>

In [ ]:
# À toi
banc_de_test = [
    ("Quelle est la défense de Charizard ?",                 "chercher_pokemon"),
    ("Combien font 12 * 12 + 1 ?",                            "calculer"),
    ("Dans mes notes de cours, c'est quoi un token ?",       "chercher_dans_mes_notes"),
    ("Quels sont les PV de Snorlax ?",                        "chercher_pokemon"),
    ("Que disent mes notes de cours sur les trois rôles ?",  "chercher_dans_mes_notes"),
    ("Combien font 1000 / 8 ?",                               "calculer"),
]

def outil_utilise(question):
    return None

def score_agent(banc):
    return None

for question, attendu in banc_de_test:
    obtenu = outil_utilise(question)
    print(f"{'✓' if obtenu == attendu else '✗'} {question[:48]:48} attendu {attendu:24} obtenu {obtenu}")
print("Score :", score_agent(banc_de_test), "/", len(banc_de_test))

In [ ]:
verifier("Exercice 9 · outil_utilise", lambda: outil_utilise("Quelle est la défense de Charizard ?") == "chercher_pokemon" and outil_utilise("Bonjour, qui es-tu ?") is None)
verifier("Exercice 9 · score 6 / 6 (mode démo)", lambda: score_agent(banc_de_test) == 6)

<details><summary>Solution</summary>

```python
banc_de_test = [
    ("Quelle est la défense de Charizard ?",                 "chercher_pokemon"),
    ("Combien font 12 * 12 + 1 ?",                            "calculer"),
    ("Dans mes notes de cours, c'est quoi un token ?",       "chercher_dans_mes_notes"),
    ("Quels sont les PV de Snorlax ?",                        "chercher_pokemon"),
    ("Que disent mes notes de cours sur les trois rôles ?",  "chercher_dans_mes_notes"),
    ("Combien font 1000 / 8 ?",                               "calculer"),
]

def outil_utilise(question):
    reponse = llm([{"role": "system", "content": SYSTEME_AGENT}, {"role": "user", "content": question}],
                  max_new_tokens=60, temperature=0)
    appel = extraire_appel(reponse)
    return appel[0] if appel else None

def score_agent(banc):
    return sum(outil_utilise(question) == attendu for question, attendu in banc)

for question, attendu in banc_de_test:
    obtenu = outil_utilise(question)
    print(f"{'✓' if obtenu == attendu else '✗'} {question[:48]:48} attendu {attendu:24} obtenu {obtenu}")
print("Score :", score_agent(banc_de_test), "/", len(banc_de_test))
```

</details>

## Exercice 10 ⭐⭐⭐ · Enchaîner deux outils

« Combien fait la vitesse de Pikachu fois 3 ? » demande **deux** outils à la suite : chercher la vitesse, puis calculer. Écris `agent_journal(question, max_etapes=4)` : la même boucle que `agent`, mais qui renvoie le couple `(reponse_finale, outils_appeles)` où `outils_appeles` est la liste des noms d'outils utilisés, dans l'ordre.

Résultat attendu : `(..."270"..., ["chercher_pokemon", "calculer"])`.

<details><summary>Indice</summary>

Recopie `agent`, ajoute `outils_appeles = []` avant la boucle, `outils_appeles.append(nom)` après chaque appel, et renvoie `(reponse, outils_appeles)` aux deux `return`.

</details>

In [ ]:
# À toi
def agent_journal(question, max_etapes=4):
    return None

reponse, outils_appeles = agent_journal("Combien fait la vitesse de Pikachu fois 3 ?") or (None, None)
print(reponse)
print("Outils appelés :", outils_appeles)

In [ ]:
verifier("Exercice 10 · deux outils dans l'ordre", lambda: outils_appeles == ["chercher_pokemon", "calculer"])
verifier("Exercice 10 · 90 × 3 = 270", lambda: "270" in reponse)
verifier("Exercice 10 · sans outil", lambda: agent_journal("Bonjour, qui es-tu ?")[1] == [])

<details><summary>Solution</summary>

```python
def agent_journal(question, max_etapes=4):
    messages = [{"role": "system", "content": SYSTEME_AGENT}, {"role": "user", "content": question}]
    outils_appeles = []
    for etape in range(1, max_etapes + 1):
        reponse = llm(messages, max_new_tokens=60, temperature=0)
        appel = extraire_appel(reponse)
        if not appel:
            return reponse, outils_appeles
        nom, args = appel
        outils_appeles.append(nom)
        resultat = executer(nom, args)
        messages.append({"role": "assistant", "content": reponse})
        messages.append({"role": "user", "content": f"Résultat de l'outil : {resultat}\nRéponds maintenant à la question."})
    return f"Je n'ai pas réussi en {max_etapes} étapes.", outils_appeles

reponse, outils_appeles = agent_journal("Combien fait la vitesse de Pikachu fois 3 ?")
print(reponse)
print("Outils appelés :", outils_appeles)   # ['chercher_pokemon', 'calculer']
```

</details>

## Exercice 11 ⭐⭐⭐ · Quiz : les bonnes pratiques avec l'IA

Pour chaque situation, réponds `True` (bonne idée) ou `False` (mauvaise idée) dans `mes_reponses`, puis écris `corriger_quiz(mes_reponses, attendues)` qui renvoie `(score, erreurs)` où `erreurs` est la liste des **numéros** (à partir de 1) des situations ratées. Les réponses attendues sont cachées dans la cellule de vérification : compare seulement après avoir répondu.

Résultat attendu : 8 / 8, et une liste d'erreurs vide.

<details><summary>Indice</summary>

Rappelle-toi les 5 règles de la leçon : vérifier, pas de données personnelles, repérer quand l'IA se trompe, apprendre plutôt que copier, rester aux commandes. `erreurs = [i for i, (r, a) in enumerate(zip(reponses, attendues), 1) if r != a]`.

</details>

In [ ]:
# À toi
situations = [
    "Je demande à un chatbot la date de naissance d'un scientifique peu connu et je la mets dans mon exposé sans vérifier.",
    "Je colle mon code qui plante et je demande d'expliquer pourquoi il plante, sans me donner la correction.",
    "Je donne au chatbot le nom complet et l'adresse d'un ami pour qu'il écrive une blague sur lui.",
    "Pour un calcul avec beaucoup de chiffres, je fais faire le calcul par Python ou un outil, pas par le modèle.",
    "Le modèle répond avec assurance à une question sur mon jeu inventé de la séance 11, donc c'est sûrement vrai.",
    "Je demande au chatbot de me poser 5 questions sur mon cours et je réponds sans regarder mes notes.",
    "Mon agent peut envoyer des e-mails : je le laisse envoyer sans me demander, ça va plus vite.",
    "Je range ma clé d'API dans les Secrets de Colab, jamais dans le code que je pousse sur GitHub.",
]
mes_reponses = [None, None, None, None, None, None, None, None]

def corriger_quiz(reponses, attendues):
    return None

In [ ]:
ATTENDUES = [False, True, False, True, False, True, False, True]
resultat_quiz = corriger_quiz(mes_reponses, ATTENDUES)
verifier("Exercice 11 · corriger_quiz", lambda: corriger_quiz([True] * 8, ATTENDUES) == (4, [1, 3, 5, 7]) and corriger_quiz(ATTENDUES, ATTENDUES) == (8, []))
verifier("Exercice 11 · 8 / 8", lambda: resultat_quiz == (8, []))
for i in (resultat_quiz[1] if resultat_quiz else []):
    print("   à revoir :", i, "·", situations[i - 1])

<details><summary>Solution</summary>

```python
situations = [
    "Je demande à un chatbot la date de naissance d'un scientifique peu connu et je la mets dans mon exposé sans vérifier.",
    "Je colle mon code qui plante et je demande d'expliquer pourquoi il plante, sans me donner la correction.",
    "Je donne au chatbot le nom complet et l'adresse d'un ami pour qu'il écrive une blague sur lui.",
    "Pour un calcul avec beaucoup de chiffres, je fais faire le calcul par Python ou un outil, pas par le modèle.",
    "Le modèle répond avec assurance à une question sur mon jeu inventé de la séance 11, donc c'est sûrement vrai.",
    "Je demande au chatbot de me poser 5 questions sur mon cours et je réponds sans regarder mes notes.",
    "Mon agent peut envoyer des e-mails : je le laisse envoyer sans me demander, ça va plus vite.",
    "Je range ma clé d'API dans les Secrets de Colab, jamais dans le code que je pousse sur GitHub.",
]
mes_reponses = [False,   # vérifier : une date précise, ça se vérifie
                True,    # apprendre plutôt que copier
                False,   # données personnelles d'un ami : jamais
                True,    # le modèle calcule mal, un outil calcule juste
                False,   # assurance ≠ vérité (hallucination)
                True,    # l'IA comme coach pour s'interroger
                False,   # rester aux commandes : demander avant les actions importantes
                True]    # la clé ne va jamais dans le code

def corriger_quiz(reponses, attendues):
    erreurs = [i for i, (r, a) in enumerate(zip(reponses, attendues), 1) if r != a]
    return len(attendues) - len(erreurs), erreurs
```

</details>

## Exercice 12 ⭐⭐⭐ · Défi : la check-list du portfolio

Ton portfolio GitHub se pilote avec un dictionnaire `projet → {case: True/False}`. Écris :
1. `progression(portfolio)` → `{"fait": ..., "total": ..., "pourcentage": ... (entier, arrondi vers le bas), "manquants": {projet: [cases non faites]}}` (les projets complets n'apparaissent pas dans `manquants`) ;
2. `prochaine_etape(portfolio)` → la première case non faite, au format `"projet → case"`, ou `"Portfolio complet !"`.

Résultat attendu sur l'exemple : 3 / 19 cases (15 %), prochaine étape `"1. Analyse d'un dataset → 3 phrases de conclusion"`. Remplace ensuite les valeurs par les tiennes : c'est ta vraie check-list.

<details><summary>Indice</summary>

Deux boucles imbriquées `for projet, cases in portfolio.items(): for case, fait in cases.items()`. Le pourcentage : `100 * fait // total`.

</details>

In [ ]:
# À toi
portfolio = {
    "1. Analyse d'un dataset": {"notebook sur GitHub": True,  "3 graphiques lisibles": True,  "3 phrases de conclusion": False, "README de 5 lignes": False},
    "2. Dashboard et modèle":  {"notebook sur GitHub": False, "capture du dashboard": False,  "score du modèle indiqué": False, "README de 5 lignes": False},
    "3. Kaggle Titanic":       {"notebook sur GitHub": False, "score Kaggle indiqué": False,  "ce que j'ai essayé (3 lignes)": False, "README de 5 lignes": False},
    "4. Assistant IA":         {"notebook sur GitHub": False, "prompt système visible": False, "exemple de conversation": False, "README de 5 lignes": False},
    "Le dépôt lui-même":       {"README d'accueil (qui je suis, les 4 projets)": False, "pas de clé d'API ni de données perso": True, "lien du dépôt testé dans un navigateur privé": False},
}

def progression(portfolio):
    return None

def prochaine_etape(portfolio):
    return None

bilan = progression(portfolio)
print(bilan)
print("Prochaine étape :", prochaine_etape(portfolio))

In [ ]:
verifier("Exercice 12 · compte", lambda: bilan["fait"] == 3 and bilan["total"] == 19 and bilan["pourcentage"] == 15)
verifier("Exercice 12 · manquants", lambda: len(bilan["manquants"]["2. Dashboard et modèle"]) == 4 and bilan["manquants"]["1. Analyse d'un dataset"] == ["3 phrases de conclusion", "README de 5 lignes"])
verifier("Exercice 12 · prochaine_etape", lambda: prochaine_etape(portfolio) == "1. Analyse d'un dataset → 3 phrases de conclusion")
complet = {p: {c: True for c in cases} for p, cases in portfolio.items()}
verifier("Exercice 12 · portfolio complet", lambda: progression(complet)["pourcentage"] == 100 and progression(complet)["manquants"] == {} and prochaine_etape(complet) == "Portfolio complet !")

Check-list finale (à faire pour de vrai) : remplace les `True` / `False` par ton état réel, relance la cellule, et travaille la « prochaine étape » jusqu'à 100 %.

<details><summary>Solution</summary>

```python
portfolio = {
    "1. Analyse d'un dataset": {"notebook sur GitHub": True,  "3 graphiques lisibles": True,  "3 phrases de conclusion": False, "README de 5 lignes": False},
    "2. Dashboard et modèle":  {"notebook sur GitHub": False, "capture du dashboard": False,  "score du modèle indiqué": False, "README de 5 lignes": False},
    "3. Kaggle Titanic":       {"notebook sur GitHub": False, "score Kaggle indiqué": False,  "ce que j'ai essayé (3 lignes)": False, "README de 5 lignes": False},
    "4. Assistant IA":         {"notebook sur GitHub": False, "prompt système visible": False, "exemple de conversation": False, "README de 5 lignes": False},
    "Le dépôt lui-même":       {"README d'accueil (qui je suis, les 4 projets)": False, "pas de clé d'API ni de données perso": True, "lien du dépôt testé dans un navigateur privé": False},
}
def progression(portfolio):
    fait = total = 0
    manquants = {}
    for projet, cases in portfolio.items():
        total += len(cases)
        fait += sum(cases.values())
        non_faites = [case for case, ok in cases.items() if not ok]
        if non_faites:
            manquants[projet] = non_faites
    return {"fait": fait, "total": total, "pourcentage": 100 * fait // total, "manquants": manquants}

def prochaine_etape(portfolio):
    for projet, cases in portfolio.items():
        for case, ok in cases.items():
            if not ok:
                return f"{projet} → {case}"
    return "Portfolio complet !"

bilan = progression(portfolio)
print(bilan)
print("Prochaine étape :", prochaine_etape(portfolio))
```

</details>

## Bravo, c'est la fin de l'atelier !

Tu as construit un agent complet sans framework : un parseur tolérant, un dispatcher, une calculatrice sécurisée, une boucle limitée en étapes qui survit aux outils cassés, un banc de test, et un enchaînement de deux outils. Il ne reste plus qu'à finir ton portfolio (exercice 12) et à le montrer.